In [4]:
#!/usr/bin/env python3
"""
Generate ChimeraX defattr file for B-factors from binding data.
Sums difference values by site and maps to sequential site numbering.
"""

import pandas as pd


In [5]:
def main():
    # Read the input files
    print("Reading input files...")
    diffs_df = pd.read_csv('../../results/summaries/tufted_duck_MHCII_binding.csv')

    # Filter to only rows where SA23 entry >= -3
    diffs_df = diffs_df[diffs_df['entry in SA23 cells'] >= -3]

    # Sum the binding values by site
    print("\nSumming binding values by site...")
    site_sums = diffs_df.groupby('site')['tufted duck MHCII binding escape'].sum().reset_index()
    site_sums.columns = ['site', 'total_binding']

    # Generate the defattr file
    output_file = 'bfactors_binding_H3_numbering.defattr'
    print(f"\nWriting defattr file to {output_file}...")
    with open(output_file, 'w') as f:
        # Write header
        f.write("# ChimeraX defattr file\n")
        f.write("#\n")
        f.write("attribute: binding\n")
        f.write("match mode: any\n")
        f.write("recipient: residues\n")
        f.write("\n")
        # Write data lines - handle non-numeric site values
        for _, row in site_sums.iterrows():
            site_value = str(row['site'])
            # Convert letters to uppercase for ChimeraX format (125a -> 125A)
            site_formatted = site_value.replace('a', 'A').replace('b', 'B').replace('c', 'C')
            bfactor = float(row['total_binding'])
            f.write(f"\t:{site_formatted}\t{bfactor:.6f}\n")

    print(f"\nDefattr file successfully created: {output_file}")
    print(f"Total residues with B-factors: {len(site_sums)}")
    print(f"\nB-factor statistics:")
    print(f"  Min: {site_sums['total_binding'].min():.6f}")
    print(f"  Max: {site_sums['total_binding'].max():.6f}")
    print(f"  Mean: {site_sums['total_binding'].mean():.6f}")
    print(f"  Median: {site_sums['total_binding'].median():.6f}")
    print("\nExample lines from defattr file:")
    print(site_sums.head(5).to_string(index=False))

if __name__ == "__main__":
    main()

Reading input files...

Summing binding values by site...

Writing defattr file to bfactors_binding_H3_numbering.defattr...

Defattr file successfully created: bfactors_binding_H3_numbering.defattr
Total residues with B-factors: 564

B-factor statistics:
  Min: -9.884300
  Max: 7.839100
  Mean: -0.059953
  Median: 0.031639

Example lines from defattr file:
site  total_binding
  -1       0.774119
  -2       0.856400
  -3       0.696254
  -4      -0.308359
  -5      -0.025276
